# Real Estate Feasibility Engine – Python App
## Step 3: Collections engine (bookings → payment-plan milestones → cash received)

**What this notebook does**
1. Adds `engine/collections.py` – the Python version of the Excel **COLLECTIONS** sheet.
2. Shows the **milestone schedule** (which quarter each CLP milestone falls due), a worked example, and quarter-by-quarter cash per product.
3. Completes the **JV landowner share** (the cost row that 'follows collections').
4. Proves the result twice: with an **independent second method** (buyer group by buyer group) and against your **Excel FINAL file**.

**Before you start:** Steps 1 and 2 must have been run once. Run cells top to bottom with **Shift + Enter**.

### Cell 1 – Connect to Google Drive and open the project folder

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/feasibility_app'
except ImportError:
    BASE = os.path.abspath('feasibility_app')
os.chdir(BASE)
for f in ['inputs.py', 'timeline.py', 'curves.py']:
    assert os.path.exists(f'engine/{f}'), f'engine/{f} missing – please run the earlier steps first.'
print('Working in:', BASE, '| engine files:', sorted(f for f in os.listdir('engine') if f.endswith('.py')))

### Cell 2 – Write the collections engine (`engine/collections.py`)
**How a milestone becomes cash**
* **Booking-linked** (e.g. *ATS = booking + 1 quarter*): every buyer pays it that many quarters after booking.
* **Construction-linked** (e.g. *first floor roof at 35%*): it falls due in the quarter construction crosses 35%.
  Buyers who booked before that pay it then; **later buyers pay it as a catch-up** (`catch_up_lag` quarters after booking – 1 by default, i.e. with the ATS).
* In a **JV** the landowner receives his share (32% in the sample) of every rupee collected.

Because milestones are grouped by type, a plan can have **any number of milestones** (e.g. 17 or 40 floors).

In [ ]:
%%writefile engine/collections.py
"""
engine/collections.py  -  STEP 3: the COLLECTIONS engine (Python version of the Excel COLLECTIONS sheet).

Bookings  ->  payment-plan (CLP) milestones  ->  cash actually received.

Rules (same as Excel):
  * Bookings in quarter t = product revenue x sales % in t.
  * Booking-linked milestone ("Booking + k quarters"): every buyer pays it k quarters after booking.
  * Construction-linked milestone ("Construction x% complete"): it falls due in the first quarter Q in which
    the product's cumulative construction progress reaches x%.
        - buyers who booked early enough pay it in quarter Q;
        - later buyers have missed it, so they pay it as a CATCH-UP, `catch_up_lag` quarters after booking.
    Formula for quarter t:  t < Q -> 0 ;  t = Q -> pct x (all bookings up to t - lag) ;  t > Q -> pct x bookings(t - lag)
  * JV: the landowner gets his revenue share of every rupee collected.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from .inputs import Project, HORIZON_Q
from .timeline import Timeline
from .curves import Curves

TOL = 1e-9


@dataclass
class MilestoneDue:
    product: str
    plan: str
    name: str
    trigger: str
    value: float
    pct: float
    due_q: int | None          # construction-linked: quarter it falls due (None for booking-linked)
    text: str                  # plain English


@dataclass
class Collections:
    bookings: dict = field(default_factory=dict)          # product -> 40 values (Rs Cr)
    booking_linked: dict = field(default_factory=dict)    # product -> cash from booking-linked milestones
    construction_linked: dict = field(default_factory=dict)
    collections: dict = field(default_factory=dict)       # product -> total cash received per quarter
    receivables: dict = field(default_factory=dict)       # product -> booked but not yet paid (cumulative)
    schedule: list = field(default_factory=list)          # list of MilestoneDue
    total: np.ndarray = None
    landowner: np.ndarray = None
    developer: np.ndarray = None
    profile: np.ndarray = None                            # share of total collections per quarter
    checks: list = field(default_factory=list)


def _shift(arr: np.ndarray, k: int) -> np.ndarray:
    """Value of arr k quarters earlier (zeros before Q1)."""
    out = np.zeros_like(arr)
    if k == 0:
        return arr.copy()
    if k < len(arr):
        out[k:] = arr[:-k]
    return out


def trigger_quarter(cons_cum: np.ndarray, x: float, cons_start: int) -> int:
    """First quarter in which cumulative construction >= x (never before construction starts)."""
    q = int(np.sum(cons_cum < x - TOL)) + 1
    return max(q, cons_start)


def build_collections(project: Project, tl: Timeline, cv: Curves) -> Collections:
    co = Collections()
    lag = project.catch_up_lag
    qs = np.arange(1, HORIZON_Q + 1)

    for p in project.active_products():
        t = tl.products[p.name]
        B = cv.revenue[p.name] * cv.sales_pct[p.name]                  # bookings
        CB = np.cumsum(B)                                              # cumulative bookings
        B_lag, CB_lag = _shift(B, lag), _shift(CB, lag)
        book, cons = np.zeros(HORIZON_Q), np.zeros(HORIZON_Q)
        plan = project.plan(p.clp_plan)
        for m in plan.milestones:
            if m.pct == 0:
                continue
            if m.trigger == "Booking + quarters":
                book += m.pct * _shift(B, int(m.value))
                text = "at booking" if m.value == 0 else f"{int(m.value)} qtr(s) after booking"
                co.schedule.append(MilestoneDue(p.name, plan.name, m.name, m.trigger, m.value, m.pct, None, text))
            else:
                Q = trigger_quarter(cv.cons_cum[p.name], m.value, t.cons_start)
                cons += m.pct * np.where(qs < Q, 0.0, np.where(qs == Q, CB_lag, B_lag))
                text = (f"construction {m.value:.0%} reached in Q{Q}" if Q <= HORIZON_Q else "NOT REACHED in model")
                co.schedule.append(MilestoneDue(p.name, plan.name, m.name, m.trigger, m.value, m.pct, Q, text))
                if Q > HORIZON_Q:
                    co.checks.append(f"[Collections] '{p.name}': milestone '{m.name}' is never reached inside the model.")
        co.bookings[p.name] = B
        co.booking_linked[p.name] = book
        co.construction_linked[p.name] = cons
        co.collections[p.name] = book + cons
        co.receivables[p.name] = CB - np.cumsum(book + cons)

        rev = cv.revenue[p.name]                                       # product checks
        if abs(B.sum() - rev) > 1e-6 or abs((book + cons).sum() - rev) > 1e-6:
            co.checks.append(f"[Collections] '{p.name}': bookings {B.sum():.4f} / collections {(book + cons).sum():.4f} "
                             f"do not both equal revenue {rev:.4f} (cash lost beyond the horizon or plan not 100%).")
        if co.receivables[p.name].min() < -1e-6:
            co.checks.append(f"[Collections] '{p.name}': cash collected before it was booked.")

    co.total = sum(co.collections.values()) if co.collections else np.zeros(HORIZON_Q)
    lo_share = project.land.lo_share if project.info.deal_type == "JV" else 0.0
    co.landowner = co.total * lo_share
    co.developer = co.total - co.landowner
    tot = co.total.sum()
    co.profile = co.total / tot if tot > 0 else np.zeros(HORIZON_Q)

    # complete every cost head that follows collections (e.g. the JV landowner share)
    for c in project.costs:
        if cv.cost_pct.get(c.key) is None:
            cv.cost_pct[c.key] = co.profile.copy() if c.applicable else np.zeros(HORIZON_Q)

    if abs(tot - sum(cv.revenue.values())) > 1e-6:
        co.checks.append("[Collections] total collections do not equal total revenue.")
    return co


def verify_by_cohorts(project: Project, tl: Timeline, cv: Curves, co: Collections) -> float:
    """Independent second method: follow every booking quarter (cohort) of buyers one by one.
    Returns the largest difference against the fast engine above (should be ~0)."""
    lag, worst = project.catch_up_lag, 0.0
    for p in project.active_products():
        t = tl.products[p.name]
        cash = np.zeros(HORIZON_Q + 60)
        for b in range(1, HORIZON_Q + 1):
            booked = cv.revenue[p.name] * cv.sales_pct[p.name][b - 1]
            if booked == 0:
                continue
            for m in project.plan(p.clp_plan).milestones:
                if m.trigger == "Booking + quarters":
                    due = b + int(m.value)
                else:
                    due = max(trigger_quarter(cv.cons_cum[p.name], m.value, t.cons_start), b + lag)
                cash[due] += booked * m.pct
        worst = max(worst, float(np.abs(cash[1:HORIZON_Q + 1] - co.collections[p.name]).max()))
    return worst


### Cell 3 – Update the Excel comparison helper (adds the COLLECTIONS reader)

In [ ]:
%%writefile engine/excel_check.py
"""
engine/excel_check.py - reads the Excel FINAL model and compares it with the Python engine.
Works by LABELS (not cell addresses), so it is robust to row positions.
"""
from __future__ import annotations
import numpy as np
import openpyxl

NQ = 40
QCOL0 = 4      # quarter columns start at column D in TIMELINE / CURVES


def _row_values(ws, r):
    return np.array([ws.cell(r, c).value or 0 for c in range(QCOL0, QCOL0 + NQ)], dtype=float)


def read_excel_curves(path: str) -> dict:
    """Returns {'sales': {product: arr}, 'cons': {...}, 'cost': {cost name: arr}, 'profiles': {...}}"""
    wb = openpyxl.load_workbook(path, data_only=True, read_only=False)
    ws = wb["CURVES"]
    out = {"sales": {}, "cons": {}, "cost": {}, "cost_order": [], "profiles": {}}
    block, section = None, None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("B.  PRODUCT"):   section = "products"; continue
        if label.startswith("C.  AGGREGATE"): section = "profiles"; continue
        if label.startswith("D.  COST HEAD"): section = "costs"; continue
        if label.startswith("E.  CURVE CHECKS"): break
        if section == "products":
            if label.startswith("   Sales % (curve"):         out["sales"][block] = _row_values(ws, r)
            elif label.startswith("   Construction % (curve"): out["cons"][block] = _row_values(ws, r)
            elif not label.startswith("   "):                   block = label
        elif section == "profiles":
            out["profiles"][label] = _row_values(ws, r)
        elif section == "costs":
            if label.startswith("   Spend %"): out["cost"][block] = _row_values(ws, r); out["cost_order"].append(block)
            elif not label.startswith("   "):   block = label
    return out


def read_excel_timeline(path: str) -> dict:
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["INPUTS"]
    prods = {}
    for r in range(1, ws.max_row + 1):          # product table: active rows have quarters in P..S
        if ws.cell(r, 5).value == "Yes" and isinstance(ws.cell(r, 16).value, (int, float)) and isinstance(ws.cell(r, 2).value, str):
            prods[ws.cell(r, 2).value] = tuple(int(ws.cell(r, c).value) for c in (16, 17, 18, 19))
    return prods


def read_excel_collections(path: str) -> dict:
    """Bookings, collections, totals, JV split and milestone due quarters from the COLLECTIONS sheet."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["COLLECTIONS"]
    out = {"bookings": {}, "collections": {}, "triggers": {}, "total": None, "landowner": None, "developer": None}
    section, block = None, None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("A.  MILESTONE"): section = "A"; continue
        if label.startswith("B.  PRODUCT"):   section = "B"; continue
        if label.startswith("C.  PROJECT"):   section = "C"; continue
        if label.startswith("D.  COLLECTION"): break
        if section == "A":
            if "   →   " in label:
                block = label.split("   →   ")[0]; out["triggers"][block] = []
            elif block and label.strip() and not label.startswith("   ") and ws.cell(r, 5).value:
                due = ws.cell(r, 6).value
                out["triggers"][block].append((label, int(due) if isinstance(due, (int, float)) else None))
        elif section == "B":
            if label.startswith("   Bookings (sales"):        out["bookings"][block] = _row_values(ws, r)
            elif label.startswith("   Collections (cash"):    out["collections"][block] = _row_values(ws, r)
            elif not label.startswith("   "):                 block = label
        elif section == "C":
            if label.startswith("Total collections (cash"):   out["total"] = _row_values(ws, r)
            elif label.startswith("   Landowner share"):      out["landowner"] = _row_values(ws, r)
            elif label.startswith("   Developer share"):      out["developer"] = _row_values(ws, r)
    return out


### Cell 4 – Run the engine chain and see the milestone schedule

In [ ]:
import importlib, numpy as np, pandas as pd
import engine.inputs, engine.timeline, engine.curves, engine.collections, engine.excel_check
for m in (engine.inputs, engine.timeline, engine.curves, engine.collections, engine.excel_check): importlib.reload(m)
from engine.inputs import *
from engine.timeline import build_timeline
from engine.curves import build_curves
from engine.collections import build_collections, verify_by_cohorts

project = sample_project()
tl = build_timeline(project)
cv = build_curves(project, tl)
co = build_collections(project, tl, cv)
problems = validate(project) + tl.errors + cv.checks + co.checks
print('✔ ALL CHECKS PASSED' if not problems else problems)
print(f'Total bookings {sum(b.sum() for b in co.bookings.values()):.2f} Cr | total collections {co.total.sum():.2f} Cr | '
      f'landowner {co.landowner.sum():.2f} Cr | developer {co.developer.sum():.2f} Cr')

print('\nMILESTONE SCHEDULE')
sched = pd.DataFrame([{'Product': s.product, 'Plan': s.plan, 'Milestone': s.name, '% of sale': f'{s.pct:.1%}', 'Falls due': s.text}
                      for s in co.schedule])
for prod, grp in sched.groupby('Product', sort=False):
    print(f'\n▶ {prod}  ({grp.Plan.iloc[0]})'); display(grp.drop(columns=['Product', 'Plan']).reset_index(drop=True))

### Cell 5 – Worked example: where does the Villa cash in Q5 come from?
Each line is calculated live from the engine – change the inputs and the explanation changes too.

In [ ]:
prod, q = 'Villas', 5
B = co.bookings[prod]; lag = project.catch_up_lag
print(f'{prod}: bookings  ' + '  '.join(f'Q{i+1}: {B[i]:.2f}' for i in range(q) if B[i]))
total = 0.0
for s in [s for s in co.schedule if s.product == prod]:
    if s.due_q is None:                                    # booking-linked
        k = int(s.value); b = q - k
        amt = s.pct * (B[b - 1] if b >= 1 else 0)
        if amt: print(f'  {s.name:32} {s.pct:5.1%} of Q{b} bookings ({B[b-1]:.2f})                = {amt:6.2f} Cr')
    else:                                                  # construction-linked
        if q == s.due_q:
            base = B[:q - lag].sum(); amt = s.pct * base
            if amt: print(f'  {s.name:32} {s.pct:5.1%} of all bookings up to Q{q-lag} ({base:.2f}) = {amt:6.2f} Cr   ← falls due now')
        elif q > s.due_q:
            amt = s.pct * B[q - lag - 1]
            if amt: print(f'  {s.name:32} {s.pct:5.1%} of Q{q-lag} bookings ({B[q-lag-1]:.2f}) = {amt:6.2f} Cr   ← catch-up')
        else: amt = 0
    total += amt
print(f'  {"TOTAL cash received in Q" + str(q):32} {"":44} = {total:6.2f} Cr   (engine: {co.collections[prod][q-1]:.2f})')

### Cell 6 – Quarter-by-quarter cash per product and the JV split (₹ Cr)

In [ ]:
Q = [f'Q{i}' for i in range(1, 23)]
rows = {}
for name in co.collections:
    rows[(name, 'Bookings')] = co.bookings[name][:22]
    rows[(name, 'Booking-linked cash')] = co.booking_linked[name][:22]
    rows[(name, 'Construction-linked cash')] = co.construction_linked[name][:22]
    rows[(name, 'Collections')] = co.collections[name][:22]
    rows[(name, 'Receivables (cumulative)')] = co.receivables[name][:22]
rows[('PROJECT', 'Total collections')] = co.total[:22]
rows[('PROJECT', 'Landowner share (JV)')] = co.landowner[:22]
rows[('PROJECT', 'Developer share')] = co.developer[:22]
display(pd.DataFrame(rows, index=Q).T.round(2))

### Cell 7 – Chart: bookings vs cash received

In [ ]:
import plotly.graph_objects as go
total_bookings = sum(co.bookings.values())
fig = go.Figure()
fig.add_bar(x=Q, y=total_bookings[:22], name='Bookings (sales value)', marker_color='#A9D08E')
fig.add_bar(x=Q, y=co.total[:22], name='Cash collected', marker_color='#5B9BD5')
fig.add_scatter(x=Q, y=np.cumsum(total_bookings)[:22] - np.cumsum(co.total)[:22], name='Receivables (sold, not yet paid)',
                mode='lines', line=dict(color='#C00000'))
fig.update_layout(barmode='group', title='Bookings vs cash collected (₹ Cr) – cash lags sales because of the payment plans', height=420)
fig.show()

### Cell 8 – Proof 1: an independent second method
`verify_by_cohorts` ignores the fast formulas and simply follows **every booking quarter's group of buyers**, milestone by milestone.
Both methods must agree.

In [ ]:
diff = verify_by_cohorts(project, tl, cv, co)
print(('✔' if diff < 1e-9 else '✖'), f'Largest difference between the two methods: {diff:.1e} Cr')
for name in co.collections:
    print(f'   {name:28} collected {co.collections[name].sum():8.3f} Cr = revenue {cv.revenue[name]:8.3f} Cr  '
          f'{"✔" if abs(co.collections[name].sum() - cv.revenue[name]) < 1e-6 else "✖"}')

### Cell 9 – Proof 2: compare with the Excel FINAL file
Uses `data/Feasibility_Engine_FINAL.xlsx` (from Step 2) – or asks you to upload it.

In [ ]:
from engine.excel_check import read_excel_collections, read_excel_curves
xl_path = 'data/Feasibility_Engine_FINAL.xlsx'
if not os.path.exists(xl_path):
    try:
        from google.colab import files
        print('Please choose Feasibility_Engine_FINAL.xlsx …'); up = files.upload(); os.replace(next(iter(up)), xl_path)
    except Exception as e:
        print('Upload skipped:', e)
if os.path.exists(xl_path):
    x, xc = read_excel_collections(xl_path), read_excel_curves(xl_path)
    tests = {
        'Bookings per product':        max(abs(co.bookings[n] - x['bookings'][n]).max() for n in co.bookings),
        'Collections per product':     max(abs(co.collections[n] - x['collections'][n]).max() for n in co.collections),
        'Total collections':           abs(co.total - x['total']).max(),
        'Landowner share':             abs(co.landowner - x['landowner']).max(),
        'Developer share':             abs(co.developer - x['developer']).max(),
        'Land cost phasing (CURVES)':  abs(cv.cost_pct['land'] - xc['cost'][xc['cost_order'][0]]).max(),
    }
    for k, v in tests.items(): print(('✔' if v < 1e-9 else '✖'), f'{k:30} largest difference {v:.1e}')
    ok = all([(s.name, s.due_q) for s in co.schedule if s.product == n and s.due_q] == [t for t in x['triggers'][n] if t[1]] for n in co.collections)
    print(('✔' if ok else '✖'), 'Milestone due quarters identical for every product')
else:
    print('Excel file not available – skipped.')

### Cell 10 – It adapts: change one input, cash re-flows (and every rupee is still collected)

In [ ]:
import copy
def run(p, label):
    t = build_timeline(p); c = build_curves(p, t); k = build_collections(p, t, c)
    cum = np.cumsum(k.total) / k.total.sum()
    q50, q90 = int(np.argmax(cum >= 0.5)) + 1, int(np.argmax(cum >= 0.9)) + 1
    ok = '✔' if not (t.errors + c.checks + k.checks) and verify_by_cohorts(p, t, c, k) < 1e-9 else '✖'
    print(f'{ok} {label:44} collected {k.total.sum():7.2f} Cr  50% by Q{q50:<3} 90% by Q{q90:<3} landowner {k.landowner.sum():6.2f} Cr')

run(project, 'Base case')
p = copy.deepcopy(project); p.catch_up_lag = 0;                          run(p, 'Catch-up paid at booking (lag 0)')
p = copy.deepcopy(project); p.catch_up_lag = 2;                          run(p, 'Catch-up paid 2 quarters after booking')
p = copy.deepcopy(project); p.products[1].clp_plan = 'CLP – 17-Floor Tower'; run(p, 'Apartments on the 17-floor plan (25 milestones)')
p = copy.deepcopy(project); p.products[1].construction_duration_q = 24;  run(p, 'Apartment construction 16 → 24 quarters')
p = copy.deepcopy(project); p.info.scenario = 'Worst Case';               run(p, 'Worst Case scenario')
p = copy.deepcopy(project); p.info.deal_type = 'Outright';                run(p, 'Outright deal (no landowner share)')

---
### ✅ Step 3 complete
`engine/` now holds `inputs.py`, `timeline.py`, `curves.py`, `collections.py` and `excel_check.py`.

**Next – Step 4:** cash flow & returns – outflows for every cost head, landowner payments with advance recovery, project and developer cash flow, IRR, NPV, peak investment, payback, and the full P&L.